In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import optuna

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

SEED = 42

In [ ]:
ROOT = Path.cwd().parent.resolve()

TRAIN = ROOT / "data/train.csv"
TEST = ROOT / "data/test.csv"
OUT = ROOT / "outputs/xgboost"
Path.mkdir(OUT , exist_ok=True)

train_df = pd.read_csv(str(TRAIN), sep=',')
test_df = pd.read_csv(str(TEST), sep=',')

X = train_df.drop(columns=["id", "Heart Disease"])
le = LabelEncoder()
y = le.fit_transform(train_df["Heart Disease"])
X_test = test_df.drop(columns="id")

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
def objective(trial):
    params = {
        # hyperparameters to tune
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 50.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 10.0),
        "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),  # L2
        "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),   # L1
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",          # or "gpu_hist" if you use GPU
            device="cuda",
            random_state=SEED,
            n_jobs=-1,
            n_estimators=2000,
            early_stopping_rounds=150,
        )
        
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=0,
        )

        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study.optimize(objective, n_trials=50)

In [ ]:
best_params = study.best_params
seeds = [2, 64, 351, 8237]
tests = []
oofs = []

for seed in seeds:
    params = {**best_params, "random_state": seed}

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    oof_seed = np.zeros(len(X))
    test_seed = np.zeros(len(X_test))

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",          # or "gpu_hist" if you use GPU
            device="cuda",
            n_jobs=-1,
            n_estimators=2000,
            early_stopping_rounds=150,
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=0
        )

        oof_seed[val_idx] += model.predict_proba(X_val)[:, 1]
        test_seed += model.predict_proba(X_test)[:, 1] / skf.n_splits

    oofs.append(oof_seed)
    tests.append(test_seed)

roc_auc_score(y, np.vstack(oofs).mean(axis=0))

In [ ]:
final_oofs = np.vstack(oofs).mean(axis=0)
final_preds = np.vstack(tests).mean(axis=0)

np.save(OUT / "xgb_oof.npy", final_oofs)
np.save(OUT / "xgb_test.npy", final_preds)

submission = pd.DataFrame({
    "id": test_df["id"],
    "Heart Disease": final_preds
})
submission.to_csv(OUT / "xgb_submission.csv", index=False)
submission